### 2026-01 yellow taxi

In [ ]:
import pandas as pd

df = pd.read_parquet(
    "../data/raw/yellow_tripdata_2026-01.parquet"
)

In [ ]:
print(df.head(5))
df.shape

   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         2  2026-01-01 00:54:04   2026-01-01 00:59:37              1.0   
1         1  2026-01-01 00:34:04   2026-01-01 00:39:47              0.0   
2         1  2026-01-01 00:57:06   2026-01-01 01:05:59              0.0   
3         2  2026-01-01 00:15:22   2026-01-01 00:58:10              4.0   
4         2  2026-01-01 00:27:13   2026-01-01 00:40:43              0.0   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           0.97         1.0                  N           239           238   
1           0.90         1.0                  N           163           162   
2           1.40         1.0                  N            43           237   
3           5.58         1.0                  N           142           209   
4           2.16         1.0                  N            88           144   

   payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  \


(3724889, 20)

In [3]:
df.columns
df.dtypes
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3724889 entries, 0 to 3724888
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

## Quality Assessment

In [4]:
df.isnull().sum() #检查缺失值
missing = df.isnull().sum()
missing_rate = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({
    "Missing Count": missing,
    "Missing Rate (%)": missing_rate
})
missing_summary

,Missing Count,Missing Rate (%)
VendorID,0,0.00
tpep_pickup_datetime,0,0.00
tpep_dropoff_datetime,0,0.00
passenger_count,1088058,29.21
trip_distance,0,0.00
RatecodeID,1088058,29.21
store_and_fwd_flag,1088058,29.21
PULocationID,0,0.00
DOLocationID,0,0.00
payment_type,0,0.00


In [5]:
df[df["passenger_count"].isna()]["VendorID"].value_counts() #找出passenger_count是空的行

VendorID
2    944635
1    139406
6      4017
Name: count, dtype: int64

In [6]:
missing_rows = df["passenger_count"].isna() #检查是不是同样的这 1,088,058 条记录，同时缺少另外 4 个字段。
df[missing_rows].isna().sum()

VendorID                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count          1088058
trip_distance                  0
RatecodeID               1088058
store_and_fwd_flag       1088058
PULocationID                   0
DOLocationID                   0
payment_type                   0
fare_amount                    0
extra                          0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge     1088058
Airport_fee              1088058
cbd_congestion_fee             0
dtype: int64

In [ ]:
df[missing_rows][
    [
        "VendorID",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "payment_type",
        "fare_amount",
        "total_amount"
    ]
].head(10)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,payment_type,fare_amount,total_amount
2636831,2,2026-01-01 00:28:47,2026-01-01 00:49:30,43,200,0,82.01,89.19
2636832,2,2026-01-01 00:54:24,2026-01-01 01:15:01,200,51,0,51.77,53.27
2636833,2,2026-01-01 00:12:31,2026-01-01 00:41:02,142,234,0,32.17,36.92
2636834,2,2026-01-01 00:48:56,2026-01-01 01:08:00,90,114,0,18.29,24.04
2636835,2,2026-01-01 00:32:09,2026-01-01 00:41:05,237,163,0,15.02,22.77
2636836,2,2026-01-01 00:24:17,2026-01-01 00:31:51,129,83,0,13.87,25.37
2636837,2,2026-01-01 00:49:31,2026-01-01 01:21:03,238,107,0,24.77,29.52
2636838,2,2026-01-01 00:39:35,2026-01-01 01:27:01,79,151,0,59.36,76.93
2636839,2,2026-01-01 00:27:04,2026-01-01 00:36:03,114,79,0,9.87,14.62
2636840,2,2026-01-01 00:44:00,2026-01-01 01:08:00,75,50,0,33.59,44.09


In [8]:
df[missing_rows]["payment_type"].value_counts()

payment_type
0    1088058
Name: count, dtype: int64

In [9]:
df[df["payment_type"] == 0].shape

(1088058, 20)

A structured missing-value pattern was identified. All 1,088,058 records with payment_type = 0 have missing values in passenger_count, RatecodeID, store_and_fwd_flag, congestion_surcharge, and Airport_fee. These records still contain complete pickup/drop-off timestamps, location IDs, payment information, fare amounts, and total amounts. Therefore, they should not be automatically removed as invalid records. Instead, their suitability should be assessed according to the specific analytical metric.

### 检查异常值--trip_distance

In [10]:
df["trip_distance"].describe()

count    3.724889e+06
mean     6.455647e+00
std      6.488855e+02
min      0.000000e+00
25%      1.000000e+00
50%      1.810000e+00
75%      3.730000e+00
max      2.690975e+05
Name: trip_distance, dtype: float64

In [11]:
(df["trip_distance"] <= 0).sum() #有多少条出租车记录的行程距离小于等于 0

125738

In [12]:
(df["trip_distance"] == 0).sum()#行程记录为0

125738

In [13]:
df["trip_distance"].quantile([0.90, 0.95, 0.99, 0.999])

0.900     8.56
0.950    12.45
0.990    19.54
0.999    29.79
Name: trip_distance, dtype: float64

99.9% 的行程距离 ≤ 29.79 miles

In [ ]:
df.nlargest(10, "trip_distance")[
    [
        "VendorID",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "PULocationID",
        "DOLocationID",
        "fare_amount",
        "total_amount"
    ]
]#找出 trip_distance 最大的 10 条出租车记录，然后查看这些异常长距离订单的关键字段。

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,PULocationID,DOLocationID,fare_amount,total_amount
3524923,2,2026-01-28 01:08:00,2026-01-28 01:16:00,269097.48,79,144,12.19,16.94
3632367,2,2026-01-30 13:00:00,2026-01-30 13:38:00,237567.19,42,130,72.99,81.95
3285750,2,2026-01-21 16:31:00,2026-01-21 16:40:00,236216.69,234,107,13.58,21.33
3131676,2,2026-01-17 10:40:00,2026-01-17 10:57:00,236183.06,137,82,22.75,34.96
2704619,2,2026-01-02 21:31:00,2026-01-02 21:55:00,221544.91,249,41,31.21,39.55
2688274,2,2026-01-02 07:07:00,2026-01-02 07:18:00,221436.73,231,161,27.61,32.36
3131406,2,2026-01-17 10:39:00,2026-01-17 10:56:00,196490.48,13,162,33.59,38.34
3571687,2,2026-01-29 07:51:00,2026-01-29 08:26:00,192081.68,33,246,60.20,64.95
2820951,2,2026-01-07 17:51:00,2026-01-07 17:56:00,184908.33,164,186,12.18,16.93
3499856,2,2026-01-27 14:46:00,2026-01-27 16:10:00,183363.75,230,74,71.01,75.76


In [15]:
(df["trip_distance"] > 100).sum()

162

In [16]:
df[df["trip_distance"] > 100][
    [
        "trip_distance",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "fare_amount",
        "total_amount"
    ]
].head(20)

,trip_distance,tpep_pickup_datetime,tpep_dropoff_datetime,fare_amount,total_amount
77980,36610.27,2026-01-02 10:20:45,2026-01-02 10:36:06,105.27,117.21
191169,100.37,2026-01-03 17:21:42,2026-01-03 17:33:31,12.80,21.06
252325,112.49,2026-01-04 13:41:17,2026-01-04 15:58:41,689.70,722.39
266099,123.18,2026-01-04 16:13:36,2026-01-04 18:27:44,797.50,807.19
290678,121.69,2026-01-05 00:01:05,2026-01-05 01:59:25,762.50,799.19
314561,122.20,2026-01-05 12:17:06,2026-01-05 14:43:36,400.00,415.24
476833,107.16,2026-01-07 11:52:34,2026-01-07 11:52:34,438.40,560.74
498532,116.06,2026-01-07 15:27:20,2026-01-07 18:12:08,380.00,381.00
565542,192.80,2026-01-08 11:25:05,2026-01-08 11:56:17,33.50,39.60
626420,106.06,2026-01-08 21:23:52,2026-01-08 23:16:43,660.30,672.01


### 检查行程时间的异常值

In [17]:
df["trip_duration_minutes"] = (
    df["tpep_dropoff_datetime"] -
    df["tpep_pickup_datetime"]
).dt.total_seconds() / 60 #计算行程时间

In [18]:
df["avg_speed_mph"] = (
    df["trip_distance"] /
    (df["trip_duration_minutes"] / 60)
)#计算平均速度

In [19]:
df.nlargest(10, "avg_speed_mph")[
    [
        "trip_distance",
        "trip_duration_minutes",
        "avg_speed_mph",
        "fare_amount",
        "total_amount"
    ]
]#看速度最快的

,trip_distance,trip_duration_minutes,avg_speed_mph,fare_amount,total_amount
16,1.54,0.0,inf,28.9,41.58
126,1.79,0.0,inf,18.4,24.15
251,1.24,0.0,inf,8.6,16.32
252,2.00,0.0,inf,11.4,17.15
253,0.83,0.0,inf,6.5,12.25
287,3.83,0.0,inf,27.5,39.90
382,5.38,0.0,inf,27.5,32.50
405,1.22,0.0,inf,10.0,18.90
511,1.69,0.0,inf,19.8,30.66
527,1.13,0.0,inf,7.9,16.38


In [20]:
(df["trip_duration_minutes"] == 0).sum()

45069

In [21]:
(df["trip_duration_minutes"] < 0).sum()

1

In [22]:
(df[(df["trip_duration_minutes"] == 0) & (df["trip_distance"] > 0)].shape[0])
#多少条记录同时满足：行程时间 = 0 分钟 AND 行驶距离 > 0

43905

### 检查重复值

In [23]:
# 检测完全重复的记录
duplicate_count = df.duplicated().sum()

print("重复记录数量：", duplicate_count)

重复记录数量： 0


### 金额异常

In [24]:
# 检查 fare_amount
print("fare_amount 负数：",
      (df["fare_amount"] < 0).sum())

print("fare_amount 等于0：",
      (df["fare_amount"] == 0).sum())

# 检查 total_amount
print("total_amount 负数：",
      (df["total_amount"] < 0).sum())

print("total_amount 等于0：",
      (df["total_amount"] == 0).sum())

fare_amount 负数： 39463
fare_amount 等于0： 2082
total_amount 负数： 39984
total_amount 等于0： 433


In [25]:
df[["fare_amount", "total_amount"]].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]
)

,fare_amount,total_amount
count,3.724889e+06,3.724889e+06
mean,2.080425e+01,2.917853e+01
std,1.892701e+01,2.258553e+01
min,-2.555200e+03,-2.560200e+03
50%,1.560000e+01,2.305000e+01
90%,4.219000e+01,5.341000e+01
95%,5.850000e+01,7.525000e+01
99%,8.140000e+01,1.047500e+02
99.9%,1.458000e+02,1.755145e+02
max,2.555200e+03,2.560200e+03


In [ ]:
df.nlargest(
    10,
    "total_amount"
)[[
    "tpep_pickup_datetime",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "payment_type"
]]#从整个数据集中找出 total_amount（订单总金额）最高的 10 条记录
#然后查看这些记录的时间、距离、基础车费、总金额和支付方式。

,tpep_pickup_datetime,trip_distance,fare_amount,total_amount,payment_type
2127025,2026-01-25 00:43:10,48.65,2555.2,2560.20,3
1230759,2026-01-15 13:56:22,0.00,2500.0,2500.00,1
1608067,2026-01-19 17:09:00,0.00,990.0,991.00,3
2167312,2026-01-26 17:27:28,128.05,899.0,911.71,2
2035988,2026-01-24 00:18:59,9.03,900.0,904.25,4
2153231,2026-01-26 13:30:11,138.54,893.4,904.11,4
84952,2026-01-02 12:36:07,0.00,850.0,850.00,1
1847335,2026-01-22 10:21:55,121.18,837.4,838.90,2
1586362,2026-01-19 13:10:40,116.46,826.2,827.70,2
2435631,2026-01-29 20:16:15,13.92,800.3,807.55,4
